# Qubit-Medic - end-to-end Colab notebook

Runs SFT warm-up + GRPO RL on a single Colab T4. Total wall-clock: ~24 hours
(SFT ~30 min, GRPO ~22 hours, eval ~30 min). The notebook is structured so
every cell is idempotent and re-runnable.

Before submitting: run from a clean Google account on a fresh T4 (the
Section 11 reproducibility gate).

## 1. Clone the repo and install

In [ ]:
%cd /content
!git clone https://github.com/qubit-medic/qubit-medic.git || (cd qubit-medic && git pull)
%cd qubit-medic
!pip install -q -r requirements.txt
!pip install -q -r requirements-train.txt
!pip install -q --no-deps unsloth

## 2. Validate the environment

All five gates must pass before going further.

In [ ]:
!python -m scripts.validate_env

## 3. Section 1.3 - format-test (existential go/no-go)

If parseable rate is below 30%, SFT is mandatory.

In [ ]:
!python -m scripts.format_test --backend unsloth --model Qwen/Qwen2.5-3B-Instruct --syndromes 10 --samples-per 3 --out data/format_test.json

## 4. Generate SFT data (5,000 syndromes, ~5 min)

In [ ]:
!python -m scripts.generate_sft_data --n 5000 --out data/sft_dataset.jsonl

## 5. SFT warm-up (~30 min on T4)

In [ ]:
!python -m scripts.train_sft \
    --dataset data/sft_dataset.jsonl \
    --output checkpoints/sft_warmup \
    --report-to none

## 6. SFT validation gate (Section 6.2)

In [ ]:
!python -m scripts.eval --adapter checkpoints/sft_warmup --episodes 100 --out data/sft_eval.json

## 7. GRPO RL training (~22 hours on T4)

Use `wandb` for live monitoring (the run will print a URL). Adjust
`--steps` if your time budget is tighter (~250 steps/hour on a T4).

In [ ]:
!wandb login  # paste your W&B API key (or skip and pass --report-to none)
!python -m scripts.train_grpo \
    --sft-checkpoint checkpoints/sft_warmup \
    --output checkpoints/grpo \
    --steps 2000 \
    --report-to wandb

## 8. Final evaluation (Section 7 + Section 10 plots)

In [ ]:
!python -m scripts.eval --adapter checkpoints/grpo --episodes 500 --out data/grpo_eval.json
!python -m scripts.baseline_policies --episodes 500 --out data/baseline_results.json
!python -m scripts.plot_results --baselines data/baseline_results.json --out-dir figures
!python -m scripts.animate_grid --frames 50

## 9. Optional: Willow real-chip cross-validation (Section 8)

In [ ]:
# Manually download from https://zenodo.org/record/13359217 and place at data/willow_d3.dem
!python -m scripts.willow_validation --dem data/willow_d3.dem --episodes 1000

## 10. Push to Hugging Face Spaces

After successful training, push the env + adapters to a Space.

In [ ]:
from huggingface_hub import HfApi, login
login()  # paste your HF token
api = HfApi()
# Replace with your Space repo id.
api.upload_folder(folder_path='.', repo_id='your-team/qubit-medic', repo_type='space')